In [1]:
import os

folders = ["src", "data"]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

with open("src/__init__.py", "w", encoding="utf-8") as f:
    pass

print("Project folders created successfully!")

Project folders created successfully!


In [2]:
%%writefile src/utils.py
import logging
import os

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)


def setup_logging(name, level=logging.INFO):
    logger = logging.getLogger(name)
    logger.setLevel(level)

    if not logger.handlers:
        handler = logging.StreamHandler()
        formatter = logging.Formatter(
            "%(asctime)s - %(name)s - %(levelname)s - %(message)s"
        )
        handler.setFormatter(formatter)
        logger.addHandler(handler)

    return logger


def ensure_directory_exists(path):
    if not path:
        return

    try:
        os.makedirs(path, exist_ok=True)
    except OSError as error:
        logging.error(f"Error creating directory {path}: {error}")
        raise

Writing src/utils.py


In [3]:
%%writefile data/input_sales.csv
OrderID,Product,Amount,Price,Customer
101,Laptop,1200.50,1200.50,Alice
102,Mouse,25.00,25.00,Bob
103,Keyboard,75.25,75.25,Alice
104,Monitor,300.00,300.00,Charlie
105,Webcam,45.75,45.75,Bob
106,External HDD,90.00,90.00,David
107,Headphones,150.00,150.00,Alice
108,USB Hub,15.50,15.50,Eve

Writing data/input_sales.csv


In [4]:
%%writefile data/input_inventory.json
[
    {
        "id": "PROD001",
        "name": "Laptop Pro",
        "category": "Electronics",
        "stock": 50,
        "details": {
            "brand": "TechCorp",
            "model": "XPS 15"
        }
    },
    {
        "id": "PROD002",
        "name": "Mechanical Keyboard",
        "category": "Peripherals",
        "stock": 120,
        "details": {
            "brand": "KeyMaster",
            "layout": "US ANSI"
        }
    },
    {
        "id": "PROD003",
        "name": "Ergonomic Mouse",
        "category": "Peripherals",
        "stock": 80,
        "details": {
            "brand": "ErgoGear",
            "dpi": 1600
        }
    }
]

Writing data/input_inventory.json


In [5]:
%%writefile src/csv_handler.py
import csv
import os

from .utils import setup_logging, ensure_directory_exists

logger = setup_logging(__name__)


class CSVHandler:
    """Read, process, and write CSV data."""

    def read_csv_as_dicts(self, file_path):
        if not os.path.exists(file_path):
            logger.error(f"CSV file not found: {file_path}")
            return None

        try:
            with open(file_path, "r", newline="", encoding="utf-8") as file:
                reader = csv.DictReader(file)
                data = list(reader)

            logger.info(f"Read {len(data)} rows from {file_path}")
            return data

        except (OSError, csv.Error) as error:
            logger.error(f"Error reading CSV: {error}")
            return None

    def process_sales_data(self, sales_data, min_sale_amount=0):
        total_sales = 0.0
        total_price = 0.0
        valid_count = 0
        filtered_sales = []

        for row in sales_data:
            try:
                amount = float(row["Amount"])
                price = float(row["Price"])

                total_sales += amount
                total_price += price
                valid_count += 1

                if amount >= min_sale_amount:
                    filtered_sales.append(row)

            except (ValueError, TypeError, KeyError):
                logger.warning(f"Skipping invalid row: {row}")

        average_price = (
            total_price / valid_count if valid_count else 0.0
        )

        summary = {
            "Total Sales": f"{total_sales:.2f}",
            "Average Item Price": f"{average_price:.2f}",
            "Filtered Sales Count": len(filtered_sales)
        }

        logger.info(f"Sales summary: {summary}")

        return {
            "filtered_sales": filtered_sales,
            "summary": summary
        }

    def write_dicts_to_csv(self, data, file_path, fieldnames):
        directory = os.path.dirname(file_path)
        ensure_directory_exists(directory)

        try:
            with open(file_path, "w", newline="", encoding="utf-8") as file:
                writer = csv.DictWriter(
                    file,
                    fieldnames=fieldnames,
                    extrasaction="ignore"
                )
                writer.writeheader()
                writer.writerows(data)

            logger.info(f"Wrote {len(data)} rows to {file_path}")
            return True

        except (OSError, csv.Error) as error:
            logger.error(f"Error writing CSV: {error}")
            return False

Writing src/csv_handler.py


In [6]:
%%writefile src/json_handler.py
import json
import os

from .utils import setup_logging, ensure_directory_exists

logger = setup_logging(__name__)


class JSONHandler:
    """Read, write, and update JSON inventory data."""

    def read_json(self, file_path):
        if not os.path.exists(file_path):
            logger.error(f"JSON file not found: {file_path}")
            return None

        try:
            with open(file_path, "r", encoding="utf-8") as file:
                data = json.load(file)

            logger.info(f"Read JSON from {file_path}")
            return data

        except json.JSONDecodeError as error:
            logger.error(f"Invalid JSON format: {error}")
            return None
        except OSError as error:
            logger.error(f"Error reading JSON: {error}")
            return None

    def write_json(self, data, file_path, indent=4):
        ensure_directory_exists(os.path.dirname(file_path))

        try:
            with open(file_path, "w", encoding="utf-8") as file:
                json.dump(data, file, indent=indent, ensure_ascii=False)

            logger.info(f"Wrote JSON to {file_path}")
            return True

        except OSError as error:
            logger.error(f"Error writing JSON: {error}")
            return False

    def update_inventory(self, inventory_data, product_id, new_stock_level):
        if not isinstance(inventory_data, list):
            logger.warning("Inventory data must be a list.")
            return False

        if not isinstance(new_stock_level, int) or new_stock_level < 0:
            logger.warning("Stock must be a non-negative integer.")
            return False

        for product in inventory_data:
            if product.get("id") == product_id:
                product["stock"] = new_stock_level
                logger.info(
                    f"Updated {product_id} stock to {new_stock_level}"
                )
                return True

        logger.warning(f"Product {product_id} not found.")
        return False

    def add_product(self, inventory_data, new_product):
        if not isinstance(inventory_data, list):
            logger.warning("Inventory data must be a list.")
            return False

        new_id = new_product.get("id")

        if not new_id:
            logger.warning("New product must have an ID.")
            return False

        if any(product.get("id") == new_id for product in inventory_data):
            logger.warning(f"Product ID {new_id} already exists.")
            return False

        inventory_data.append(new_product)
        logger.info(f"Added product: {new_product.get('name', 'N/A')}")
        return True

Writing src/json_handler.py


In [7]:
%%writefile main.py
import os

from src.csv_handler import CSVHandler
from src.json_handler import JSONHandler
from src.utils import setup_logging, ensure_directory_exists

logger = setup_logging(__name__)


def main():
    data_dir = "data"
    ensure_directory_exists(data_dir)

    input_csv_path = os.path.join(data_dir, "input_sales.csv")
    output_csv_path = os.path.join(
        data_dir, "output_filtered_sales.csv"
    )
    input_json_path = os.path.join(data_dir, "input_inventory.json")
    output_json_path = os.path.join(
        data_dir, "output_updated_inventory.json"
    )

    # CSV workflow
    logger.info("--- Starting CSV Processing ---")
    csv_handler = CSVHandler()
    sales_data = csv_handler.read_csv_as_dicts(input_csv_path)

    if sales_data is None:
        logger.error("Could not read the sales CSV.")
    else:
        results = csv_handler.process_sales_data(
            sales_data,
            min_sale_amount=100.0
        )

        filtered_sales = results["filtered_sales"]
        summary = results["summary"]

        logger.info(f"Summary: {summary}")

        fieldnames = [
            "OrderID", "Product", "Amount", "Price", "Customer"
        ]

        summary_row = {
            "OrderID": "SUMMARY",
            "Product": "Total/Avg",
            "Amount": summary["Total Sales"],
            "Price": summary["Average Item Price"],
            "Customer": f"Count: {summary['Filtered Sales Count']}"
        }

        output_rows = filtered_sales + [summary_row]

        csv_handler.write_dicts_to_csv(
            output_rows,
            output_csv_path,
            fieldnames
        )

    # JSON workflow
    logger.info("--- Starting JSON Processing ---")
    json_handler = JSONHandler()
    inventory = json_handler.read_json(input_json_path)

    if inventory is None:
        logger.error("Could not read the inventory JSON.")
    elif not isinstance(inventory, list):
        logger.error("Inventory JSON must contain a list of products.")
    else:
        json_handler.update_inventory(inventory, "PROD001", 60)
        json_handler.update_inventory(inventory, "PROD003", 75)

        new_product = {
            "id": "PROD004",
            "name": "Gaming Headset",
            "category": "Audio",
            "stock": 90,
            "details": {
                "brand": "SoundBlaster",
                "features": ["Noise Cancelling", "RGB"]
            }
        }

        json_handler.add_product(inventory, new_product)
        json_handler.write_json(
            inventory,
            output_json_path,
            indent=2
        )

    logger.info("--- All Workflows Completed ---")


if __name__ == "__main__":
    main()

Writing main.py


In [8]:
!python main.py

2026-10-10 19:55:26,033 - __main__ - INFO - --- Starting CSV Processing ---
2026-10-10 19:55:26,033 - INFO - --- Starting CSV Processing ---
2026-10-10 19:55:26,033 - src.csv_handler - INFO - Read 8 rows from data/input_sales.csv
2026-10-10 19:55:26,033 - INFO - Read 8 rows from data/input_sales.csv
2026-10-10 19:55:26,033 - src.csv_handler - INFO - Sales summary: {'Total Sales': '1902.00', 'Average Item Price': '237.75', 'Filtered Sales Count': 3}
2026-10-10 19:55:26,033 - INFO - Sales summary: {'Total Sales': '1902.00', 'Average Item Price': '237.75', 'Filtered Sales Count': 3}
2026-10-10 19:55:26,033 - __main__ - INFO - Summary: {'Total Sales': '1902.00', 'Average Item Price': '237.75', 'Filtered Sales Count': 3}
2026-10-10 19:55:26,033 - INFO - Summary: {'Total Sales': '1902.00', 'Average Item Price': '237.75', 'Filtered Sales Count': 3}
2026-10-10 19:55:26,033 - src.csv_handler - INFO - Wrote 4 rows to data/output_filtered_sales.csv
2026-10-10 19:55:26,033 - INFO - Wrote 4 rows to

In [9]:
import csv
import pandas as pd
from IPython.display import display

with open(
    "data/output_filtered_sales.csv",
    "r",
    newline="",
    encoding="utf-8"
) as file:
    csv_data = list(csv.DictReader(file))

display(pd.DataFrame(csv_data))

,OrderID,Product,Amount,Price,Customer
0,101,Laptop,1200.50,1200.50,Alice
1,104,Monitor,300.00,300.00,Charlie
2,107,Headphones,150.00,150.00,Alice
3,SUMMARY,Total/Avg,1902.00,237.75,Count: 3


In [10]:
import json
from IPython.display import display

with open(
    "data/output_updated_inventory.json",
    "r",
    encoding="utf-8"
) as file:
    updated_inventory = json.load(file)

display(pd.json_normalize(updated_inventory))

,id,name,category,stock,details.brand,details.model,details.layout,details.dpi,details.features
0,PROD001,Laptop Pro,Electronics,60,TechCorp,XPS 15,NaN,NaN,NaN
1,PROD002,Mechanical Keyboard,Peripherals,120,KeyMaster,NaN,US ANSI,NaN,NaN
2,PROD003,Ergonomic Mouse,Peripherals,75,ErgoGear,NaN,NaN,1600.0,NaN
3,PROD004,Gaming Headset,Audio,90,SoundBlaster,NaN,NaN,NaN,"[Noise Cancelling, RGB]"


In [11]:
from google.colab import files

files.download("data/output_filtered_sales.csv")
files.download("data/output_updated_inventory.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>